In [ ]:
from apeGmsh import apeGmsh
from apeGmsh import Numberer, FEMData
import numpy as np
import openseespy.opensees as ops
from pathlib import Path

In [2]:
model_iges_path = Path(r'C:\Users\nmora\Github\apeGmsh\acad') / 'Frame3D.iges'
assert model_iges_path.exists(), model_iges_path

m1 = apeGmsh(
    model_name='Frame3D_story',
    verbose=True
)
m1.begin()

m1.model.io.load_iges(
    file_path=model_iges_path,
    highest_dim_only=False
)

m1.model.queries.remove_duplicates(tolerance=1)
m1.model.queries.make_conformal(tolerance=1.0)
m1.model.queries.remove_duplicates(tolerance=1)
m1.model.sync()

m1.model.selection.select_points(on_plane=("z", 0, 1e-3)).to_physical("base_supports")
m1.model.selection.select_curves(aligned="z").to_physical("columns")
m1.model.selection.select_surfaces(on_plane=("z", 3000, 1e-3)).to_physical("slab")

col_tops = m1.model.selection.select_points(on_plane=("z", 3000, 1e-3))
print(f"Column-top nodes: {col_tops.to_tags()}")

(
    m1.mesh.sizing
        .set_size_sources(from_points=True, extend_from_boundary=True)
        .set_global_size(3000)
        .set_size(col_tops.to_tags(), 1000)
)

m1.mesh.generation.generate(dim=2)

(
    m1.mesh.editing
        .remove_duplicate_nodes()
        .remove_duplicate_elements()
)


fem_data = m1.mesh.queries.get_fem_data()


Gmsh version: 4.15.1
[Model] loaded IGES <- Frame3D.iges  {2: 6, 1: 36, 0: 72}
[Model] remove_duplicates(tolerance=1): merged {0: 12, 1: 7} entities (before={0: 48, 1: 36, 2: 6, 3: 0}, after={0: 36, 1: 29, 2: 6, 3: 0})
[Model] make_conformal(dims=[0, 1, 2], tolerance=1.0): entity delta={} (before={0: 36, 1: 29, 2: 6, 3: 0}, after={0: 36, 1: 29, 2: 6, 3: 0})
[Model] remove_duplicates(tolerance=1): merged {} entities (before={0: 36, 1: 29, 2: 6, 3: 0}, after={0: 36, 1: 29, 2: 6, 3: 0})
[Model] OCC kernel synchronised
[Selection] select dim=0 -> 12 / 36 entities
[PhysicalGroups] add(dim=0, entities=[26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48]) -> pg_tag=1, name='base_supports'
[Selection] select dim=1 -> 12 / 29 entities
[PhysicalGroups] add(dim=1, entities=[25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36]) -> pg_tag=2, name='columns'
[Selection] select dim=2 -> 6 / 6 entities
[PhysicalGroups] add(dim=2, entities=[1, 2, 3, 4, 5, 6]) -> pg_tag=3, name='slab'
[Selection] select dim=0 -> 

In [3]:
m1.model.viewer()



[brep_scene] Built in 0.07s  (3 actors, 59 entities)
  Mesh generate : 0.000s  (existing)
  dim0 points   : 0.033s  (24)
  dim1 curves   : 0.013s  (29)
  dim2 surfaces : 0.017s  (6)
  dim3 volumes  : 0.000s  (0)
[viewer] closed — 2 physical group(s) written, 1 picks in working set


In [4]:
m1.physical.summary()


--- Physical Groups ---
                     name  n_entities                                     entity_tags
dim pg_tag                                                                           
0   1       base_supports          12  26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48
1   2             columns          12  25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36
    9                test           1                                              25
2   10               slab           6                                1, 2, 3, 4, 5, 6


name  n_entities  \
dim pg_tag                              
0   1       base_supports          12   
1   2             columns          12   
    9                test           1   
2   10               slab           6   

                                               entity_tags  
dim pg_tag                                                  
0   1       26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48  
1   2       25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36  
    9                                                   25  
2   10                                    1, 2, 3, 4, 5, 6

In [5]:
m1.mesh.viewer()

[MeshViewer] auto-filter: requested dims=[1, 2, 3], meshed dims=[0, 1, 2], skipping empty [3]

[mesh_scene] Built in 0.34s  (2 actors, 131 nodes)
  Entities: {1: 29, 2: 6}
  Node setup    : 0.000s
  Actor creation: 0.039s
  Remainder     : 0.301s
[Mesh] get_fem_data(dim=2) -> 131 nodes, 176 elements, bw=120


In [6]:
import openseespy.opensees as ops
import pandas as pd

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

linear_time_series_tag = 1
ops.timeSeries('Linear', linear_time_series_tag)

# Nodes
for node_id, coords in zip(fem_data.node_ids, fem_data.node_coords):
    ops.node(int(node_id), *coords)

# Boundary conditions
base_nodes = fem_data.physical.get_nodes(dim=0, tag=1)['tags']
for node_id in base_nodes:
    ops.fix(int(node_id), 1, 1, 1, 1, 1, 1)

# Section
plate_section = 1
ops.section('ElasticMembranePlateSection', plate_section, 200000, 0.3, 300, 0)
column_section = 2
# section('Elastic', secTag, E_mod, A, Iz, Iy, G_mod, Jxx, alphaY=None, alphaZ=None)
ops.section('Elastic', column_section, 200000, 300*300, 300**4/12, 300**4/12, 200000/2, 300**4/12)

# geomTransf('Linear', transfTag, *vecxz, '-jntOffset', *dI, *dJ)
linear_tag = 1
ops.geomTransf('Linear', linear_tag, *[1,0,0])

# Elements
for elem_id, (node_i, node_j) in zip(fem_data.physical.get_elements(dim=1, tag=2)['element_ids'], fem_data.physical.get_elements(dim=1, tag=2)['connectivity']):
    # element('elasticBeamColumn', eleTag, *eleNodes, secTag, transfTag, <'-mass', mass>, <'-cMass'>, <'-release', releaseCode>)
    ops.element('elasticBeamColumn', int(elem_id), int(node_i), int(node_j), column_section, linear_tag)

for elem_id, (node_i, node_j, node_k) in zip(fem_data.physical.get_elements(dim=2, tag=3)['element_ids'], fem_data.physical.get_elements(dim=2, tag=3)['connectivity']):
    ops.element('ShellDKGT', int(elem_id), int(node_i), int(node_j), int(node_k), plate_section)

# Load
shell_pattern_tag = 1
ops.pattern('Plain', shell_pattern_tag, linear_time_series_tag)

q = -1.0  # N/mm² pressure (downward)

slab_tag = fem_data.physical.get_tag(2, "slab")
slab_elems = fem_data.physical.get_elements(2, slab_tag)

# Build a nodal load accumulator
nodal_fz = {}

for conn in slab_elems['connectivity']:
    ni, nj, nk = int(conn[0]), int(conn[1]), int(conn[2])

    # Node indices for coordinate lookup
    idx_i = np.searchsorted(fem_data.node_ids, ni)
    idx_j = np.searchsorted(fem_data.node_ids, nj)
    idx_k = np.searchsorted(fem_data.node_ids, nk)

    pi = fem_data.node_coords[idx_i]
    pj = fem_data.node_coords[idx_j]
    pk = fem_data.node_coords[idx_k]

    # Triangle area = 0.5 * ||(pj - pi) × (pk - pi)||
    area = 0.5 * np.linalg.norm(np.cross(pj - pi, pk - pi))

    # Consistent nodal load: q * A / 3 per node
    fz = q * area / 3.0

    for n in (ni, nj, nk):
        nodal_fz[n] = nodal_fz.get(n, 0.0) + fz

# Apply accumulated forces
for nid, fz in nodal_fz.items():
    ops.load(nid, 0.0, 0.0, fz, 0.0, 0.0, 0.0)

# load(nodeTag, *loadValues)
ops.load(104, *[0,0,-1000,0,0,0])

# Analysis
ops.constraints('Transformation')
ops.numberer('RCM')
ops.system('BandGeneral')
ops.test('NormDispIncr', 1.0e-6, 10)
ops.algorithm('Newton')
ops.integrator('LoadControl', 0.1)
ops.analysis('Static')

ok = ops.analyze(10)
if ok != 0:
    print("Analysis failed to converge")
else:
    print("Analysis completed successfully")

AttributeError: 'FEMData' object has no attribute 'node_ids'

In [ ]:
disp = np.array([ops.nodeDisp(int(nid)) for nid in fem_data.node_ids])  # (N, 6)

# ─── Shell thickness (from section definition) ──────────────────────
h = 300.0  # mm

# ─── Slab element stress extraction ─────────────────────────────────
slab_tag  = fem_data.physical.get_tag(2, "slab")
slab_elem = fem_data.physical.get_elements(2, slab_tag)
n_slab    = len(slab_elem['element_ids'])

sig_xx = np.zeros(n_slab)
sig_yy = np.zeros(n_slab)
sig_xy = np.zeros(n_slab)
sig_vm = np.zeros(n_slab)

for i, eid in enumerate(slab_elem['element_ids']):
    # 'forces' → [N11, N22, N12, M11, M22, M12, Q13, Q23] per gauss pt
    resp = ops.eleResponse(int(eid), 'forces')
    if len(resp) >= 3:
        N11, N22, N12 = resp[0], resp[1], resp[2]
        sig_xx[i] = N11 / h
        sig_yy[i] = N22 / h
        sig_xy[i] = N12 / h
        # von Mises (plane stress)
        sig_vm[i] = np.sqrt(
            sig_xx[i]**2
          - sig_xx[i] * sig_yy[i]
          + sig_yy[i]**2
          + 3.0 * sig_xy[i]**2
        )

# ─── Build Results with all fields ──────────────────────────────────
from apeGmsh import Results

results = Results.from_fem(
    fem_data,
    point_data={"Displacement": disp},
    cell_data={
        "VonMises":  sig_vm,
        "Stress_xx": sig_xx,
        "Stress_yy": sig_yy,
        "Stress_xy": sig_xy,
    },
)
print(results.summary())

Results: 'results'
  Mesh: 170 nodes, 274 elements (250 primary + 24 extra)
  Point fields: Displacement
  Cell fields:  VonMises, Stress_xx, Stress_yy, Stress_xy
  Physical groups: 3


In [ ]:
# Open the interactive pyGmshViewer (direct in-memory, no temp files)
# Time-step slider appears automatically for time-series data.
results.viewer(blocking=False)


# Other options:
   # non-blocking (Jupyter subprocess)
# results.to_vtu("output.vtu")     # export single step
# results.to_pvd("modes")          # export time-series (.pvd + .vtu)

In [ ]:
m1.end()